## Consumer — Event Hub (Kafka) -> crypto_bronze.ticks_stream
Stores the raw JSON string as-is (no `from_json`/`StructType` here) so new producer fields never get silently dropped. Parsing with schema evolution happens downstream in silver.

In [0]:
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("bronze_schema", "team_crypto_bronze")
dbutils.widgets.text("eventhub_namespace", "evhua5816bd")
dbutils.widgets.text("eventhub_name", "crypto-ticks")
dbutils.widgets.text("secret_scope", "team-crypto-scope")
dbutils.widgets.text("checkpoint_base", "abfss://lena066636@dlsua5816bd.dfs.core.windows.net/_checkpoints")


In [0]:
from pyspark.sql import functions as F

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
eventhub_namespace = dbutils.widgets.get("eventhub_namespace")
eventhub_name = dbutils.widgets.get("eventhub_name")
secret_scope = dbutils.widgets.get("secret_scope")
checkpoint_path = dbutils.widgets.get("checkpoint_base") + "/ticks_stream"

connection_str = dbutils.secrets.get(scope=secret_scope, key="eventhub-connection-string")
bronze_table = f"{catalog}.{bronze_schema}.ticks_stream"


In [0]:
jaas_config = (
    'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="$ConnectionString" password="{connection_str}";'
)

kafka_options = {
    "kafka.bootstrap.servers": f"{eventhub_namespace}.servicebus.windows.net:9093",
    "subscribe": eventhub_name,
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": jaas_config,
    "startingOffsets": "earliest",
}


In [0]:
df_raw = spark.readStream.format("kafka").options(**kafka_options).load()


In [0]:
df_bronze = (df_raw
    .withColumn("value", F.col("value").cast("string"))
    .withColumn("source", F.lit("eventhub"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("load_date", F.current_date())
    .select("value", "source", "ingestion_timestamp", "load_date"))


In [0]:
(df_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(processingTime="20 seconds")
    .toTable(bronze_table))
